In [1]:
import os
import numpy as np
import pandas as pd

In [3]:
df_train = pd.read_csv('<RADCHEST_CT_DATASET_DIR>/imgtrain_Abnormality_and_Location_Labels.csv')
df_val = pd.read_csv('<RADCHEST_CT_DATASET_DIR>/imgvalid_Abnormality_and_Location_Labels.csv')
df_test = pd.read_csv('<RADCHEST_CT_DATASET_DIR>/imgtest_Abnormality_and_Location_Labels.csv')
df = pd.concat([df_train, df_val, df_test], ignore_index=True)

In [4]:
df

,NoteAcc_DEID,bandlike_or_linear*left_upper,bandlike_or_linear*left_mid,bandlike_or_linear*left_lower,bandlike_or_linear*right_upper,bandlike_or_linear*right_mid,bandlike_or_linear*right_lower,bandlike_or_linear*right_lung,bandlike_or_linear*left_lung,bandlike_or_linear*lung,...,other_path*esophagus,other_path*stomach,other_path*intestine,other_path*liver,other_path*gallbladder,other_path*kidney,other_path*adrenal_gland,other_path*spleen,other_path*pancreas,other_path*other_location
0,trn24737,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,trn08439,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,trn11613,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,trn15638,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,trn03087,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3625,tst29123,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3626,tst34883,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3627,tst34029,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3628,tst34239,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [5]:
# Extract unique disease names from column headers
disease_columns = df.columns[1:]  # Exclude 'NoteAcc_DEID'
unique_diseases = list(set([c.split('*')[0] for c in disease_columns]))
print(f'Found {len(unique_diseases)} unique diseases')

# Create binary labels dataframe
binary_labels = pd.DataFrame()
binary_labels['NoteAcc_DEID'] = df['NoteAcc_DEID']

# For each disease, check if any location has value 1
for disease in sorted(unique_diseases):
    # Get all columns for this disease
    disease_cols = [c for c in disease_columns if c.split('*')[0] == disease]
    # Binary label is 1.0 if any location is 1, else 0.0
    binary_labels[disease] = df[disease_cols].max(axis=1).astype(float)

print(f'Binary labels shape: {binary_labels.shape}')
binary_labels.to_csv('<RADCHEST_CT_DATASET_DIR>/binary_labels.csv', index=False)
binary_labels.head()

Found 84 unique diseases
Binary labels shape: (3630, 85)


,NoteAcc_DEID,air_trapping,airspace_disease,aneurysm,arthritis,aspiration,atelectasis,atherosclerosis,bandlike_or_linear,breast_implant,...,septal_thickening,soft_tissue,staple,stent,sternotomy,suture,tracheal_tube,transplant,tree_in_bud,tuberculosis
0,trn24737,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1,trn08439,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,trn11613,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,trn15638,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,trn03087,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [6]:
mapping_to_ct_rate_pathologies = {
    "Medical material": [
    "gi_tube", "heart_valve_replacement", "suture", "stent", "clip",
    "hardware", "chest_tube", "pacemaker_or_defib", "tracheal_tube",
    "catheter_or_port", "staple", "sternotomy", "breast_implant"],
    "Arterial wall calcification": ["atherosclerosis"],
    "Cardiomegaly": ["cardiomegaly"],
    "Pericardial effusion": ["pericardial_effusion"],
    "Coronary artery wall calcification": ["coronary_artery_disease"],
    "Hiatal hernia": ["hernia"],
    "Lymphadenopathy": ["lymphadenopathy"],
    "Emphysema": ["emphysema"],
    "Atelectasis": ["atelectasis"],
    "Lung nodule": ["nodule", "nodulegr1cm", "scattered_nod", "granuloma"],
    "Lung opacity": ["opacity"],
    "Pulmonary fibrotic sequela": ["fibrosis", "scarring", "honeycombing"],
    "Pleural effusion": ["pleural_effusion", "hemothorax"],
    "Mosaic attenuation pattern": ["air_trapping"],
    "Peribronchial thickening": ["bronchial_wall_thickening"],
    "Consolidation": ["consolidation"],
    "Bronchiectasis": ["bronchiectasis", "bronchiolectasis"],
    "Interlobular septal thickening": ["septal_thickening"],
}

In [7]:
PATHOLOGIES_LIST = [
    "Medical material",
    "Arterial wall calcification",
    "Cardiomegaly",
    "Pericardial effusion",
    "Coronary artery wall calcification",
    "Hiatal hernia",
    "Lymphadenopathy",
    "Emphysema",
    "Atelectasis",
    "Lung nodule",
    "Lung opacity",
    "Pulmonary fibrotic sequela",
    "Pleural effusion",
    "Mosaic attenuation pattern",
    "Peribronchial thickening",
    "Consolidation",
    "Bronchiectasis",
    "Interlobular septal thickening",
]

In [9]:
# Map detailed binary labels to broader disease categories using mapping2
broad_labels = pd.DataFrame()
broad_labels['NoteAcc_DEID'] = binary_labels['NoteAcc_DEID']

for broad_category, detailed_diseases in mapping_to_ct_rate_pathologies.items():
    # Find which detailed diseases exist in binary_labels columns
    existing_cols = [d for d in detailed_diseases if d in binary_labels.columns]
    if existing_cols:
        # If any detailed disease is 1, the broad category is 1
        broad_labels[broad_category] = binary_labels[existing_cols].max(axis=1).astype(float)
    else:
        print(f"Warning: No columns found for '{broad_category}': {detailed_diseases}")
        broad_labels[broad_category] = 0.0

print(f'Broad labels shape: {broad_labels.shape}')
print(f'Columns: {broad_labels.columns.tolist()}')
broad_labels.to_csv('<RADCHEST_CT_DATASET_DIR>/broad_labels.csv', index=False)
broad_labels.head()

Broad labels shape: (3630, 19)
Columns: ['NoteAcc_DEID', 'Medical material', 'Arterial wall calcification', 'Cardiomegaly', 'Pericardial effusion', 'Coronary artery wall calcification', 'Hiatal hernia', 'Lymphadenopathy', 'Emphysema', 'Atelectasis', 'Lung nodule', 'Lung opacity', 'Pulmonary fibrotic sequela', 'Pleural effusion', 'Mosaic attenuation pattern', 'Peribronchial thickening', 'Consolidation', 'Bronchiectasis', 'Interlobular septal thickening']


,NoteAcc_DEID,Medical material,Arterial wall calcification,Cardiomegaly,Pericardial effusion,Coronary artery wall calcification,Hiatal hernia,Lymphadenopathy,Emphysema,Atelectasis,Lung nodule,Lung opacity,Pulmonary fibrotic sequela,Pleural effusion,Mosaic attenuation pattern,Peribronchial thickening,Consolidation,Bronchiectasis,Interlobular septal thickening
0,trn24737,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0
1,trn08439,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
2,trn11613,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
3,trn15638,1.0,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
4,trn03087,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
